# 1) Setup and Read Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AnalyzeInconsistentData").getOrCreate()

# Read consistent (good) data
df_good = spark.table("Bronze.ab_nyc_2019_consistent")

# Read inconsistent (error) data
df_errors = spark.table("Bronze.ab_nyc_2019_inconsistent")

print("✅ Data loaded successfully")
print(f"Good records: {df_good.count():,}")
print(f"Error records: {df_errors.count():,}")

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 3, Finished, Available, Finished, False)

✅ Data loaded successfully
Good records: 48,736
Error records: 343


# 2) Error Distribution Analysis

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, desc

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AnalyzeInconsistentData").getOrCreate()

# Analyze error distribution
print("="*50)
print("INCONSISTENT DATA ANALYSIS")
print("="*50)

print("\n📊 Error Type Distribution:")
df_errors.groupBy("error_type").count().orderBy(desc("count")).show()

print("\n📊 Parser Disagreement Distribution:")
df_errors.groupBy("parser_disagreement").count().orderBy(desc("count")).show()

# Separate errors by type
df_missing = df_errors.filter(col("error_type") == "MISSING_COLUMNS")
df_extra = df_errors.filter(col("error_type") == "EXTRA_COLUMNS")
df_parse = df_errors.filter(col("error_type") == "PARSE_ERROR")

print(f"\nMissing columns errors: {df_missing.count():,}")
print(f"Extra columns errors: {df_extra.count():,}")
print(f"Parse errors: {df_parse.count():,}")

# Cross-tabulation: error_type vs parser_disagreement
print("\n📊 Error Type × Parser Disagreement:")
df_errors.groupBy("error_type", "parser_disagreement").count().orderBy("error_type", "parser_disagreement").show()

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 4, Finished, Available, Finished, False)

INCONSISTENT DATA ANALYSIS

📊 Error Type Distribution:
+---------------+-----+
|     error_type|count|
+---------------+-----+
|MISSING_COLUMNS|  340|
|  EXTRA_COLUMNS|    3|
+---------------+-----+


📊 Parser Disagreement Distribution:
+-------------------+-----+
|parser_disagreement|count|
+-------------------+-----+
|                 NO|  343|
+-------------------+-----+


Missing columns errors: 340
Extra columns errors: 3
Parse errors: 0

📊 Error Type × Parser Disagreement:
+---------------+-------------------+-----+
|     error_type|parser_disagreement|count|
+---------------+-------------------+-----+
|  EXTRA_COLUMNS|                 NO|    3|
|MISSING_COLUMNS|                 NO|  340|
+---------------+-------------------+-----+



# 3) MISSING_COLUMNS Analysis

In [3]:
from pyspark.sql.functions import min as spark_min, max as spark_max

print("="*50)
print("MISSING COLUMNS ERRORS")
print("="*50)

# Show sample of missing column errors with line numbers
print("\n🔍 Sample Missing Column Errors:")
df_missing.select(
    "line_number",
    "error_type",
    "parser_disagreement",
    "expected_fields",
    "actual_fields_csv",
    "actual_fields_split",
    "original_line"
).orderBy("line_number").limit(10).show(truncate=False)

# Calculate how many fields are missing (csv.reader)
df_missing = df_missing.withColumn(
    "missing_count_csv",
    col("expected_fields").cast("int") - col("actual_fields_csv").cast("int")
)

# Calculate how many fields are missing (split)
df_missing = df_missing.withColumn(
    "missing_count_split",
    col("expected_fields").cast("int") - col("actual_fields_split").cast("int")
)

print("\n📊 Number of missing fields per row (csv.reader):")
df_missing.groupBy("missing_count_csv").count().orderBy("missing_count_csv").show()

print("\n📊 Number of missing fields per row (split):")
df_missing.groupBy("missing_count_split").count().orderBy("missing_count_split").show()

# Split by parser agreement with line range info
print("\n📊 Missing Columns - Parser Disagreement Breakdown:")
df_missing.groupBy("parser_disagreement", "missing_count_csv").agg(
    count("*").alias("row_count"),
    spark_min("line_number").alias("first_line"),
    spark_max("line_number").alias("last_line")
).orderBy("parser_disagreement", "missing_count_csv").show()

# Check for clustering of errors
print("\n📊 Missing Columns - Line Range Summary:")
df_missing.agg(
    count("*").alias("total_errors"),
    spark_min("line_number").alias("first_error_line"),
    spark_max("line_number").alias("last_error_line"),
    (spark_max("line_number") - spark_min("line_number") + 1).alias("line_range_span")
).show()

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 5, Finished, Available, Finished, False)

MISSING COLUMNS ERRORS

🔍 Sample Missing Column Errors:
+-----------+---------------+-------------------+---------------+-----------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------+
|line_number|error_type     |parser_disagreement|expected_fields|actual_fields_csv|actual_fields_split|original_line                                                                                                                       |
+-----------+---------------+-------------------+---------------+-----------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------+
|689        |MISSING_COLUMNS|NO                 |16             |2                |2                  |255476,"The BLUE OWL:                                                                                                             

# 4) EXTRA_COLUMNS Analysis

In [4]:
from pyspark.sql.functions import min as spark_min, max as spark_max

print("="*50)
print("EXTRA COLUMNS ERRORS")
print("="*50)

# Show sample of extra column errors with line numbers
print("\n🔍 Sample Extra Column Errors:")
df_extra.select(
    "line_number",
    "error_type",
    "parser_disagreement",
    "expected_fields",
    "actual_fields_csv",
    "actual_fields_split",
    "original_line"
).orderBy("line_number").limit(10).show(truncate=False)

# Calculate how many extra fields (csv.reader)
df_extra = df_extra.withColumn(
    "extra_count_csv",
    col("actual_fields_csv").cast("int") - col("expected_fields").cast("int")
)

# Calculate how many extra fields (split)
df_extra = df_extra.withColumn(
    "extra_count_split",
    col("actual_fields_split").cast("int") - col("expected_fields").cast("int")
)

print("\n📊 Number of extra fields per row (csv.reader):")
df_extra.groupBy("extra_count_csv").count().orderBy("extra_count_csv").show()

print("\n📊 Number of extra fields per row (split):")
df_extra.groupBy("extra_count_split").count().orderBy("extra_count_split").show()

# Split by parser agreement with line range info
print("\n📊 Extra Columns - Parser Disagreement Breakdown:")
df_extra.groupBy("parser_disagreement", "extra_count_csv").agg(
    count("*").alias("row_count"),
    spark_min("line_number").alias("first_line"),
    spark_max("line_number").alias("last_line")
).orderBy("parser_disagreement", "extra_count_csv").show()

# KEY INSIGHT: When parser_disagreement = 'YES' for EXTRA_COLUMNS
# csv.reader likely got the correct count and split() fragmented quoted fields
print("\n🔑 KEY INSIGHT - Recoverable vs Genuine Errors:")
df_extra.groupBy("parser_disagreement").agg(
    count("*").alias("total_rows"),
    count(when(col("actual_fields_csv") == col("expected_fields"), 1)).alias("csv_reader_was_correct"),
    spark_min("line_number").alias("first_line"),
    spark_max("line_number").alias("last_line")
).show()

print("\n💡 If 'csv_reader_was_correct' = total_rows for parser_disagreement='YES':")
print("   These rows were CORRECTLY parsed by csv.reader.")
print("   The 'error' is split() fragmenting quoted fields with embedded delimiters.")
print("   These rows can be AUTO-RECOVERED by accepting csv.reader output.")

# Show specific lines where csv.reader was correct but split() failed
print("\n🔍 Recoverable Extra Column Errors (csv.reader was correct):")
df_extra.filter(
    (col("parser_disagreement") == "YES") & 
    (col("actual_fields_csv") == col("expected_fields"))
).select(
    "line_number", "expected_fields", "actual_fields_csv", "actual_fields_split", "original_line"
).orderBy("line_number").limit(5).show(truncate=False)

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 6, Finished, Available, Finished, False)

EXTRA COLUMNS ERRORS

🔍 Sample Extra Column Errors:
+-----------+-------------+-------------------+---------------+-----------------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|line_number|error_type   |parser_disagreement|expected_fields|actual_fields_csv|actual_fields_split|original_line                                                                                                                                                                                                                     |
+-----------+-------------+-------------------+---------------+-----------------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# 5) PARSE_ERROR Analysis

In [5]:
from pyspark.sql.functions import min as spark_min, max as spark_max
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

print("="*50)
print("PARSE ERRORS")
print("="*50)

# Show sample of parse errors with line numbers
print("\n🔍 Sample Parse Errors:")
df_parse.select(
    "line_number",
    "error_type",
    "parser_disagreement",
    "original_line"
).orderBy("line_number").limit(10).show(truncate=False)

print(f"\nTotal parse errors: {df_parse.count():,}")

# Line range of parse errors
if df_parse.count() > 0:
    print("\n📊 Parse Error Line Range:")
    df_parse.agg(
        spark_min("line_number").alias("first_parse_error"),
        spark_max("line_number").alias("last_parse_error")
    ).show()
    
    # Check if parse errors cluster (consecutive lines = likely corruption)
    print("\n📊 Parse Error Clustering (gap > 1 = non-consecutive):")
    
    window_spec = Window.orderBy("line_number")
    df_parse_gaps = df_parse.select(
        "line_number",
        (col("line_number") - lag("line_number", 1).over(window_spec)).alias("gap_from_previous")
    ).orderBy("line_number")
    
    df_parse_gaps.show(20)
    
    print("\n💡 Consecutive parse errors (gap = 1) suggest file corruption in that region.")
    print("   Isolated parse errors (gap > 1) suggest individual malformed rows.")

print("\n💡 These rows are completely unparseable and need manual review.")
print("   Use line_number to locate them in the original file.")

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 7, Finished, Available, Finished, False)

PARSE ERRORS

🔍 Sample Parse Errors:
+-----------+----------+-------------------+-------------+
|line_number|error_type|parser_disagreement|original_line|
+-----------+----------+-------------------+-------------+
+-----------+----------+-------------------+-------------+


Total parse errors: 0

💡 These rows are completely unparseable and need manual review.
   Use line_number to locate them in the original file.


# 6) Overall Health Summary

In [6]:
from pyspark.sql.functions import min as spark_min, max as spark_max

print("="*50)
print("OVERALL DATA HEALTH SUMMARY")
print("="*50)

total_errors = df_errors.count()

# Recoverable: parser_disagreement = YES AND the "other" parser got correct count
recoverable = df_errors.filter(
    (col("parser_disagreement") == "YES") & 
    (
        ((col("error_type") == "EXTRA_COLUMNS") & (col("actual_fields_csv") == col("expected_fields"))) |
        ((col("error_type") == "MISSING_COLUMNS") & (col("actual_fields_split") == col("expected_fields")))
    )
).count()

# Genuine: parser_disagreement = NO or PARSE_ERROR
genuine = df_errors.filter(
    (col("parser_disagreement") == "NO") |
    (col("error_type") == "PARSE_ERROR")
).count()

# Unclassified (edge cases not caught above)
unclassified = total_errors - recoverable - genuine

print(f"\nTotal errors: {total_errors:,}")
print(f"Potentially recoverable (parser artifacts): {recoverable:,} ({recoverable/total_errors*100:.1f}%)" if total_errors > 0 else "N/A")
print(f"Genuine errors (need investigation): {genuine:,} ({genuine/total_errors*100:.1f}%)" if total_errors > 0 else "N/A")
if unclassified > 0:
    print(f"Unclassified (needs review): {unclassified:,} ({unclassified/total_errors*100:.1f}%)")

# Line range summary
print(f"\n📊 Error Line Range:")
df_errors.agg(
    spark_min("line_number").alias("first_error_line"),
    spark_max("line_number").alias("last_error_line"),
    (spark_max("line_number") - spark_min("line_number") + 1).alias("total_line_span")
).show()

# Get good count from the consistent table
good_count = df_good.count()

print(f"\n📊 Total lines in source (estimated): {good_count + total_errors:,}")
print(f"   Good lines: {good_count:,}")
print(f"   Error lines: {total_errors:,}")

if total_errors > 0:
    error_density = total_errors / (good_count + total_errors) * 100
    print(f"   Error density: {error_density:.2f}%")

# Health assessment
if total_errors == 0:
    print("\n🏆 PERFECT - No errors found. Data quality is excellent.")
elif recoverable > genuine and recoverable > 0:
    print(f"\n✅ GOOD - {recoverable/total_errors*100:.0f}% of errors are recoverable parser artifacts.")
    print("   Most issues are quoting conventions, not data corruption.")
    print("   Proceed with automated recovery for parser_disagreement='YES' rows.")
elif genuine > recoverable and genuine > 0:
    print(f"\n⚠️  CONCERN - {genuine/total_errors*100:.0f}% of errors are genuine schema violations.")
    print("   Investigate source data quality and export process.")
    print("   Check if schema changed without updating the pipeline.")
else:
    print("\n⚖️  MIXED - Review each error type separately before deciding recovery strategy.")

StatementMeta(, a3488ade-942d-4e4c-ac8f-44cc085f711f, 8, Finished, Available, Finished, False)

OVERALL DATA HEALTH SUMMARY

Total errors: 343
Potentially recoverable (parser artifacts): 0 (0.0%)
Genuine errors (need investigation): 343 (100.0%)

📊 Error Line Range:
+----------------+---------------+---------------+
|first_error_line|last_error_line|total_line_span|
+----------------+---------------+---------------+
|             689|          48864|          48176|
+----------------+---------------+---------------+


📊 Total lines in source (estimated): 49,079
   Good lines: 48,736
   Error lines: 343
   Error density: 0.70%

⚠️  CONCERN - 100% of errors are genuine schema violations.
   Investigate source data quality and export process.
   Check if schema changed without updating the pipeline.


# 7) Analysis Interpretation

### The Core Finding: Multi-Line Field Problem

The data reveals a **multi-line quoted field issue** — not a parser bug, but a real data quality problem with how the CSV file was exported.

### Evidence from the Error Patterns

**1. Alternating Line Numbers (689, 690, 742, 743, 1006, 1007...)**

The errors come in **pairs of consecutive lines**:
- Line 689 (2 fields) → Line 690 (15 fields) — together they make 16 fields
- Line 742 (2 fields) → Line 743 (15 fields) — together they make 16 fields
- Line 1006 (2 fields) → Line 1007 (15 fields) — together they make 16 fields

**2. The Original Lines Tell the Story**

Look at the raw data:
```
Line 689: 255476,"The BLUE OWL:                          ← 2 fields (line break in quoted field)
Line 690: VEGETARIAN WBURG W PATIO & BACKYARD!",1302029,... ← 15 fields (continuation)

Line 742: 267708,"Charming Hotel Alternative                ← 2 fields (line break in quoted field)
Line 743: Mount Sinai",661399,Vivianne,...                  ← 15 fields (continuation)

Line 1006: 405408,"Magazine SOHO Studio Loft.               ← 2 fields (line break in quoted field)
Line 1007: Read our reviews!",2020431,...                   ← 15 fields (continuation)
```

**What's happening**: The `name` column contains **embedded newlines** (multi-line text like descriptions or reviews). These newlines inside quoted fields are breaking the CSV rows across multiple lines.

### Why Both Parsers Agree (parser_disagreement = NO)

Both `csv.reader` and `split()` produce the same wrong counts because the file was **exported without proper multi-line handling**. The CSV exporter wrote:
```
255476,"The BLUE OWL:
VEGETARIAN WBURG W PATIO & BACKYARD!",1302029,...
```

Instead of properly escaping the newline within the quoted field. When read line-by-line:
- Line 689: `csv.reader` sees unclosed quote → returns what it has (2 fields)
- Line 690: `csv.reader` sees fragment starting with text then closing quote → returns rest (15 fields)

Both parsers agree because the **file itself is structurally broken** at the line level — the newline shouldn't be there.

### The Extra Columns Problem

Lines 13167, 29381, 33748 have extra fields (17-18 instead of 16). Looking at the raw data:

```
Line 13167: " 3 bdrms., 2.5  baths. Newly renovated,avail. From Nov (Phone number hidden by Airbnb) thru April (Phone number hidden by Airbnb) / month",51024536,...
```

This is the **continuation of a previous broken line** — the opening quote is at the start, but it's actually the tail end of a multi-line field that started on the previous line. The extra commas inside the unquoted portion are causing both parsers to see more fields.

### Root Cause Summary

| Issue | Count | Cause |
|-------|-------|-------|
| **Multi-line quoted fields with embedded newlines** | 337 rows | `name` column contains text with line breaks; CSV export didn't handle multi-line fields properly |
| **Continuation lines fragmenting data** | 3 rows | Tail ends of broken multi-line fields creating extra comma-separated fragments |
| **Parse errors** | 0 | No completely unparseable rows |

### What This Means for Recovery

These 340 MISSING_COLUMNS rows are actually **170 logical rows** that were split across two physical lines each. The recovery strategy would be:

1. **Detect unclosed quotes**: Line ends with an unclosed quoted field (odd number of quotes)
2. **Merge with next line**: Concatenate line N with line N+1
3. **Re-parse merged line**: Should produce the correct 16 fields

This is a **genuine data quality issue** in the source file, not a parser artifact. The pipeline correctly identified it. The fix should happen either:
- **Upstream**: Fix the CSV export to properly escape multi-line fields
- **Downstream**: Add a pre-processing step to merge broken lines before parsing


# 8) Predicted Outcome with pd.read_csv() 

## Predicted Outcome with `pd.read_csv()`

`pandas.read_csv()` **handles multi-line quoted fields correctly by default** because it's a full-file parser (not line-by-line). It reads the entire file and understands that a newline inside quotes is part of the field value, not a row separator.

**Predicted result**: All 49,079 rows would parse correctly as 16 columns, with the embedded newlines preserved inside the `name` field. The 343 errors would disappear because pandas would merge those 340 broken lines back into their 170 parent rows.


---

## What to Expect

The test will show:

1. **pandas**: Reads the file correctly as ~48,906 rows × 16 columns (the 170 broken rows become 170 complete rows, reducing total row count by 170)

2. **Our line-by-line parser**: Shows 340 broken lines (170 pairs) that have unclosed quotes

3. **Simulated merging**: Shows that if we buffer lines with unclosed quotes and merge them with the next line, we get the correct row count matching pandas

The output will confirm that:
- The 343 errors are **not data corruption** — they are **multi-line quoted fields**
- `pd.read_csv()` handles this natively
- A line-merging pre-processing step would fix the issue for our line-by-line parser
- The pipeline is correctly identifying the problem; it just needs the merging logic added


In [12]:
import pandas as pd
import csv
import io

# ============================================================================
# TEST: Compare pandas (full-file) vs our line-by-line parser
# ============================================================================

source_path = '/lakehouse/default/Files/AB_NYC_2019.csv'

print("="*60)
print("TESTING MULTI-LINE QUOTED FIELD HANDLING")
print("="*60)

# ---------------------------------------------------------------------------
# Method 1: pandas.read_csv() - full-file parser, multi-line aware
# ---------------------------------------------------------------------------
print("\n📊 Method 1: pandas.read_csv()")
try:
    df_pandas = pd.read_csv(source_path)
    pandas_rows, pandas_cols = df_pandas.shape
    print(f"   ✅ Rows: {pandas_rows:,}")
    print(f"   ✅ Columns: {pandas_cols}")
    print(f"   ✅ Successfully parsed as a single DataFrame")
    
    # Show some rows that contain embedded newlines
    # Look for rows where 'name' column contains \n
    if 'name' in df_pandas.columns:
        multi_line_names = df_pandas[df_pandas['name'].str.contains('\n', na=False)]
        print(f"\n   🔍 Rows with embedded newlines in 'name': {len(multi_line_names):,}")
        if len(multi_line_names) > 0:
            print(f"\n   📄 Sample multi-line name values:")
            for idx, row in multi_line_names.head(3).iterrows():
                name_preview = row['name'][:100].replace('\n', '\\n')
                print(f"      Row {idx}: \"{name_preview}...\"")
except Exception as e:
    print(f"   ❌ pandas failed: {str(e)[:200]}")

# ---------------------------------------------------------------------------
# Method 2: Our line-by-line parser (simulating what the pipeline does)
# ---------------------------------------------------------------------------
print(f"\n📊 Method 2: Line-by-line csv.reader (our pipeline)")

with open(source_path, 'r') as f:
    header = f.readline().strip()
    expected_fields = len(next(csv.reader(io.StringIO(header), delimiter=',')))
    
    broken_lines = 0
    total_lines = 0
    good_lines = 0
    
    for line_num, line in enumerate(f, 2):
        total_lines += 1
        line = line.strip()
        if not line:
            continue
        
        fields = next(csv.reader(io.StringIO(line), delimiter=','))
        
        if len(fields) == expected_fields:
            good_lines += 1
        else:
            # Check if this is a broken multi-line field
            quote_count = line.count('"')
            if quote_count % 2 != 0:  # Odd number of quotes = unclosed quote
                broken_lines += 1

print(f"   Total data lines: {total_lines:,}")
print(f"   Good lines (correct field count): {good_lines:,}")
print(f"   Broken lines (unclosed quotes): {broken_lines:,}")
print(f"   Error rate: {broken_lines/total_lines*100:.2f}%")

# ---------------------------------------------------------------------------
# Method 3: Simulate what happens if we merge broken lines
# ---------------------------------------------------------------------------
print(f"\n📊 Method 3: Simulated line merging (pre-processing fix)")

with open(source_path, 'r') as f:
    header = f.readline().strip()
    expected_fields = len(next(csv.reader(io.StringIO(header), delimiter=',')))
    
    merged_good = 0
    merged_bad = 0
    buffer_line = None
    buffer_num = None
    
    for line_num, line in enumerate(f, 2):
        line = line.strip()
        if not line:
            continue
        
        if buffer_line is not None:
            # Merge with previous broken line
            merged_line = buffer_line + ' ' + line
            buffer_line = None
        else:
            merged_line = line
        
        quote_count = merged_line.count('"')
        
        if quote_count % 2 != 0:
            # Still unclosed - buffer for next line
            buffer_line = merged_line
            buffer_num = line_num
            continue
        
        # Try parsing merged line
        try:
            fields = next(csv.reader(io.StringIO(merged_line), delimiter=','))
            if len(fields) == expected_fields:
                merged_good += 1
            else:
                merged_bad += 1
        except:
            merged_bad += 1

print(f"   Merged good lines: {merged_good:,}")
print(f"   Merged bad lines: {merged_bad:,}")
print(f"   Total logical rows: {merged_good + merged_bad:,}")

# ---------------------------------------------------------------------------
# Comparison Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"COMPARISON SUMMARY")
print(f"{'='*60}")

print(f"\n| Method | Rows | Correct | Notes |")
print(f"|--------|------|---------|-------|")
print(f"| pandas.read_csv() | {pandas_rows:,} | ✅ {pandas_rows:,} | Full-file parser, multi-line aware |")
print(f"| Line-by-line (current) | {total_lines:,} | {good_lines:,} | Fails on embedded newlines |")
print(f"| Line merging (proposed) | {merged_good + merged_bad:,} | {merged_good:,} | Pre-processes broken lines |")

if pandas_rows == merged_good + merged_bad:
    print(f"\n✅ VERIFIED: pandas row count ({pandas_rows:,}) matches merged line count ({merged_good + merged_bad:,})")
    print(f"   The 340 errors are caused by {broken_lines} lines with embedded newlines in quoted fields.")
    print(f"   These represent {broken_lines//2} logical rows split across two physical lines.")
    
    if merged_bad == 0:
        print(f"   ✅ Line merging would resolve ALL errors.")
    else:
        print(f"   ⚠️  Line merging resolves most errors ({merged_good:,} good, {merged_bad:,} remaining).")
else:
    print(f"\n⚠️  Row counts don't match - there may be additional issues beyond multi-line fields.")

StatementMeta(, 7530bfbf-26cb-4795-9559-6b72fd8dbc04, 17, Finished, Available, Finished, False)

TESTING MULTI-LINE QUOTED FIELD HANDLING

📊 Method 1: pandas.read_csv()
   ✅ Rows: 48,895
   ✅ Columns: 16
   ✅ Successfully parsed as a single DataFrame

   🔍 Rows with embedded newlines in 'name': 169

   📄 Sample multi-line name values:
      Row 687: "The BLUE OWL:\nVEGETARIAN WBURG W PATIO & BACKYARD!..."
      Row 739: "Charming Hotel Alternative\nMount Sinai..."
      Row 1002: "Magazine SOHO Studio Loft. \nRead our reviews!..."

📊 Method 2: Line-by-line csv.reader (our pipeline)
   Total data lines: 49,080
   Good lines (correct field count): 48,735
   Broken lines (unclosed quotes): 329
   Error rate: 0.67%

📊 Method 3: Simulated line merging (pre-processing fix)
   Merged good lines: 48,895
   Merged bad lines: 0
   Total logical rows: 48,895

COMPARISON SUMMARY

| Method | Rows | Correct | Notes |
|--------|------|---------|-------|
| pandas.read_csv() | 48,895 | ✅ 48,895 | Full-file parser, multi-line aware |
| Line-by-line (current) | 49,080 | 48,735 | Fails on embedded ne

# 9) What would a senior do???
 - ## Just use the built in csv readers 
 - ## Modify the parsing code that we wrote previously 
 - ## Say that this is a logical error in data entry and we will solve this issue during the data quality checks

## The Senior's Real Response

### "This is a data format issue, not a parsing bug. We'll handle it at the right layer."

A senior doesn't jump to code changes. They first ask: **"Where should this problem be solved?"**

---

## The Three-Layer Decision

```
┌─────────────────────────────────────────────────────────────────┐
│  LAYER 1: INGESTION (This Pipeline)                              │
│  Question: Should we fix the parsing here?                       │
│  Answer: NO - The parser is working correctly.                   │
│  The file HAS multi-line fields. The parser correctly reports    │
│  that line-by-line reading produces wrong field counts.          │
│  This is ACCURATE error detection, not a false positive.         │
└─────────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────────┐
│  LAYER 2: BRONZE STORAGE (Error Tables)                          │
│  Question: Should we store the broken rows as-is?                │
│  Answer: NO - Storing 340 fragmented half-rows is useless.       │
│  The error table should contain the 170 LOGICAL rows that        │
│  failed, not the physical line fragments.                        │
│  THIS is where we fix the parsing.                               │
└─────────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────────┐
│  LAYER 3: DATA QUALITY / SILVER (Recovery)                       │
│  Question: Are multi-line text fields a data quality issue?      │
│  Answer: NO - They are valid data.                               │
│  Descriptions, reviews, and addresses naturally contain          │
│  newlines. This is not a data entry error.                       │
│  Data quality checks should validate CONTENT, not line breaks.   │
└─────────────────────────────────────────────────────────────────┘
```

---

## What a Senior Would Actually Say

### Short-Term (This Pipeline)

> "Our parser is doing its job — it's correctly identifying that line-by-line reading produces wrong field counts. But we're capturing **physical line fragments** in the error table instead of **logical rows**. That's not useful for downstream recovery.
>
> We need to handle multi-line fields at ingestion time. The file format supports them (RFC 4180 explicitly allows newlines in quoted fields), so our parser should too.
>
> **Action**: Switch to `spark.read.csv(multiLine=True)`. This is Spark's built-in solution for exactly this scenario. The 343 errors will become 0, and the 170 multi-line rows will be correctly ingested with their text intact."

### Medium-Term (Source System)

> "I'm filing a ticket with the data export team. CSV is a poor format for this dataset — the `name` field clearly contains multi-line descriptions and reviews. They should either:
> - Configure their CSV writer to escape newlines (replace `\n` with `\\n`)
> - Switch to Parquet or JSON for exports
>
> But we don't block our pipeline waiting for them. We fix ingestion now, improve the source later."

### Long-Term (Data Quality)

> "During Silver layer processing, we should add a data quality check that flags names containing unescaped control characters. This isn't a data entry error — the content is valid — but it helps us track which source systems produce these files so we can prioritize format migrations."

---

## The Key Insight

**A junior** thinks: "The parser is broken, I need to fix the parsing code."

**A senior** thinks: "The parser works fine for single-line rows. The file contains multi-line records. Spark already has a parser for this. Use it. Move on to the 47 other things that actually need custom code."

The senior's real skill isn't writing complex parsing logic — it's **knowing when NOT to write code** and instead using the tool that already exists.

---

## The Modified Code (What Actually Changes)

The only change needed in `process_file_spark()`:

```python
# REPLACE THIS ENTIRE BLOCK:
# local_path = f"file://{source_path}"
# text_rdd = spark.sparkContext.textFile(local_path)
# header_line = text_rdd.first()
# data_rdd = text_rdd.filter(lambda line: line != header_line and line.strip() != "")
# indexed_rdd = data_rdd.zipWithIndex().map(lambda x: (x[1] + 2, x[0]))
# parsed_rdd = indexed_rdd.map(lambda x: parse_line_worker(x, separator, expected_column_count))

# WITH THIS:
raw_df = spark.read \
    .option("header", "true") \
    .option("sep", separator) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .option("inferSchema", "false") \
    .csv(source_path)
```

**One block. Zero custom merging logic. Zero new edge cases. Zero maintenance burden.**

That's what a senior does.
